<a href="https://colab.research.google.com/github/karsuvanper/llm-chunking-strategies/blob/main/llm_chunking_strategies.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

if "COLAB_GPU" in os.environ:
  print("Setting up Colab GPU support")
  #!pip install -U torch #requires torch 2.1.1+(for efficient implementation)
  !pip install PyMuPDF #for reading pdf with python
  !pip install tqdm #for progress bars
  #!pip install sentence-transformers #for embedding models
  !pip install accelerate #for quantization model loading
  !pip install bitsandbytes #for quantizing models (less storage space)
  !pip install flash-attn --no-build-isolation #for faster attention mechanism = faster LLM

In [ ]:
#download pdf
import os
import requests

#get pdf document
pdf_path="/content/drive/MyDrive/Human-Nutrition-2020.pdf"

#download the pdf if not exist

if not os.path.exists(pdf_path):
  print("file doesn't exist,downloading..")

  url="https://www.zambiancu.org/1zRead/RevillaMarie%20Kainoa-Human-Nutrition-2020.pdf"

  #local name to save the file
  filename=pdf_path

  #send a GET request to the url
  response=requests.get(url)

  #check if request was suscessfull
  if response.status_code ==200:
    with open(filename,"wb") as file:
      file.write(response.content)
      print(f"File downloaded and saved as {filename}")
  else:
    print(f"failed to download the file.status code:{response.status_code}")
else:
  print(f"file {pdf_path} already exist")

In [ ]:
import fitz
from tqdm.auto import tqdm

def text_formatter(text:str)->str:
  '''performs minor formatting on text.'''
  cleaned_text=text.replace("\n"," ").strip()

  return cleaned_text

def open_and_read_pdf(pdf_path:str)->list[dict]:
  '''
  opens a pdf file, reads its text content page by page, and collects statistics.

  params:
  pdf_path(str): the file path to the pdf document to be opened and read.

  returns:
  list[dict]:
  A list of dictionaries, each containing the page number
  (adjusted),character count,word count,sentence count,token count and the extracted text for
  each page.
  '''
  doc = fitz.open(pdf_path)
  pages_and_texts=[]
  for page_number,page in tqdm(enumerate(doc)):
    text=page.get_text()
    text=text_formatter(text)
    pages_and_texts.append({
        "page_number":page_number-41,
        "page_char_count":len(text),
        "page_word_count":len(text.split()),
        "page_sentence_count_raw":len(text.split(". ")),
        "page_token_count":len(text)/4,
        "text":text
    })

  return pages_and_texts

pages_and_texts=open_and_read_pdf(pdf_path)
pages_and_texts[:2]

step3:Testing 5 chunking startegies:fixed, recursive,semantic,structural and llm based.

In [ ]:
def chunk_text(text:str, chunk_size:int = 500)->list:

    '''
    splits text into chunks of approx,"chunk size" characters,
    '''
    chunks=[]
    current_chunk=''
    words=text.split()

    for word in words:
      if len(current_chunk)+len(word)+1<=chunk_size:
        current_chunk+=(word+' ')
      else:
        chunks.append(current_chunk.strip())
        current_chunk=word+' '

    if current_chunk:
      chunks.append(current_chunk.strip())

    return chunks

def chunk_pdf_pages(pages_and_texts:list,chunk_size:int=500)->list[dict]:

  '''
  takes psf pages with text and splits them into chunks.

  Returns a list of dicts with page_number, chunk_index, and chunk_text.

  '''

  all_chunks=[]

  for page in tqdm(pages_and_texts):
    page_number=page['page_number']
    page_text=page['text']

    chunks=chunk_text(page_text,chunk_size=chunk_size)


    for i,chunk in enumerate(chunks):
      all_chunks.append({
          "page_number":page_number,
          "chunk_index":i,
          "chunk_char_count":len(chunk),
          "chunk_word_count":len(chunk.split()),
          "chunk_token_count":len(chunk)/4,
          "chunk_text":chunk
      })
  return all_chunks

chunked_pages=chunk_pdf_pages(pages_and_texts,chunk_size=500)
print(f"Total chunks:{len(chunked_pages)}")
print(f"first chunk(page {chunked_pages[0]['page_number']}):{chunked_pages[0]['chunk_text'][:200]}....")

In [ ]:
import random,textwrap

def _scattered_indices(n:int,k:int,jitter_frac:float=0.08)->list[int]:

  if k<=0:
    return []

  if k==1:
    return [random.randrange(n)]

  anchors=[int(round(i*(n-1)/(k-1))) for i in range(k)]
  out,seen =[],set()
  radius=max(1,int(n*jitter_frac))
  for a in anchors:
    lo,hi= max(0, a-radius),min(n-1,a+radius)
    j = random.randint(lo,hi)
    if j not in seen:
      out.append(j); seen.add(j)
  while len(out)<k:
    r=random.randrange(n)
    if r not in seen:
      out.append(r);seen.add(r)
  return out

def _draw_boxed_chunk(c: dict, wrap_at:int=96)-> str:

  header=(
      f"chunk p{c['page_number']}.idx{c['chunk_index']}"
      f"chars{c['chunk_char_count']}.words {c['chunk_word_count']}. ~tokens {c['chunk_token_count']}"
  )

  wrapped_lines=textwrap.wrap(c['chunk_text'],width=wrap_at, break_long_words=False, replace_whitespace=False)

  content_width = max ([0, *map(len, wrapped_lines)])

  box_width = max(len(header), content_width+2)

  top    = "┏" + "━" * box_width + "┓"
  hline  = "┃" + header.ljust(box_width) + "┃"
  sep    = "┣" + "━" * box_width + "┫"
  body   = "\n".join("┃ " + line.ljust(box_width - 2) + " ┃" for line in wrapped_lines) or \
             ("┃ " + "".ljust(box_width - 2) + " ┃")
  bottom = "┗" + "━" * box_width + "┛"

  return "\n".join([top, hline, sep, body, bottom])

def show_random_chunks(pages_and_texts: list, chunk_size: int = 500, k: int = 5, seed: int | None = 42):
    if seed is not None:
        random.seed(seed)

    all_chunks = chunk_pdf_pages(pages_and_texts, chunk_size=chunk_size)

    if not all_chunks:
        print("No chunks to display.")
        return

    idxs = _scattered_indices(len(all_chunks), k)

    print(f"showing{len(idxs)} scatttered random chunks out of {len(all_chunks)} total:\n")

    for i,idx in enumerate(idxs):
        print(f"#{i}")
        print(_draw_boxed_chunk(all_chunks[idx]))
        print()

#-----RUN---
assert 'pages_and_texts' in globals(), "pages_and_texts = open_and_read_pdf(pdf_path) first."
show_random_chunks(pages_and_texts, chunk_size=500, k=5, seed=42)



semantic chunking

In [ ]:
!pip -q install --upgrade "sentence-transformers==3.0.1" "transformers<5,>=4.41" scikit-learn nltk

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import nltk
nltk.download('punkt', quiet=True)

# Load once globally
semantic_model = SentenceTransformer("all-MiniLM-L6-v2")

def semantic_chunk_text(text: str, similarity_threshold: float = 0.8, max_tokens: int = 500) -> list:
    """
    Splits text into semantic chunks based on sentence similarity and max token length.
    """
    sentences = nltk.sent_tokenize(text)
    if not sentences:
        return []

    embeddings = semantic_model.encode(sentences)

    chunks = []
    current_chunk = [sentences[0]]
    current_embedding = embeddings[0]

    for i in range(1, len(sentences)):
        sim = cosine_similarity([current_embedding], [embeddings[i]])[0][0]
        chunk_token_count = len(" ".join(current_chunk)) // 4

        if sim >= similarity_threshold and chunk_token_count < max_tokens:
            current_chunk.append(sentences[i])
            current_embedding = np.mean([current_embedding, embeddings[i]], axis=0)
        else:
            chunks.append(" ".join(current_chunk))
            current_chunk = [sentences[i]]
            current_embedding = embeddings[i]

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

from tqdm.auto import tqdm

def semantic_chunk_pdf_pages(pages_and_texts: list,
                             similarity_threshold: float = 0.8,
                             max_tokens: int = 500) -> list[dict]:
    """
    Takes PDF pages with text and splits them into semantic chunks.

    Returns a list of dicts with page_number, chunk_index, and chunk_text.
    """
    all_chunks = []

    for page in tqdm(pages_and_texts, desc="Semantic chunking pages"):
        page_number = page["page_number"]
        page_text = page["text"]

        chunks = semantic_chunk_text(page_text,
                                     similarity_threshold=similarity_threshold,
                                     max_tokens=max_tokens)

        for i,chunk in enumerate(chunks):
          all_chunks.append({
              "page_number": page_number,
              "chunk_index": i,
              "chunk_char_count": len(chunk),
              "chunk_word_count": len(chunk.split()),
              "chunk_token_count": len(chunk) // 4,
              "chunk_text": chunk
          }
          )

    return all_chunks

In [ ]:
import nltk
nltk.download('punkt_tab')

semantic_chunked_pages = semantic_chunk_pdf_pages(pages_and_texts,
                                                 similarity_threshold=0.75,
                                                 max_tokens=500)

print(f"Total semantic chunks: {len(semantic_chunked_pages)}")
print(f"First semantic chunk (page {semantic_chunked_pages[0]['page_number']}):")
print(semantic_chunked_pages[0]['chunk_text'][:200] + "...")

In [ ]:
import random,textwrap

def _scattered_indices(n:int,k:int,jitter_frac:float=0.08)->list[int]:

  if k<=0:
    return []

  if k==1:
    return [random.randrange(n)]

  anchors=[int(round(i*(n-1)/(k-1))) for i in range(k)]
  out,seen =[],set()
  radius=max(1,int(n*jitter_frac))
  for a in anchors:
    lo,hi= max(0, a-radius),min(n-1,a+radius)
    j = random.randint(lo,hi)
    if j not in seen:
      out.append(j); seen.add(j)
  while len(out)<k:
    r=random.randrange(n)
    if r not in seen:
      out.append(r);seen.add(r)
  return out

def _draw_boxed_chunk(c: dict, wrap_at:int=96)-> str:

  header=(
      f"chunk p{c['page_number']}.idx{c['chunk_index']}"
      f"chars{c['chunk_char_count']}.words {c['chunk_word_count']}. ~tokens {c['chunk_token_count']}"
  )

  wrapped_lines=textwrap.wrap(c['chunk_text'],width=wrap_at, break_long_words=False, replace_whitespace=False)

  content_width = max ([0, *map(len, wrapped_lines)])

  box_width = max(len(header), content_width+2)

  top    = "┏" + "━" * box_width + "┓"
  hline  = "┃" + header.ljust(box_width) + "┃"
  sep    = "┣" + "━" * box_width + "┫"
  body   = "\n".join("┃ " + line.ljust(box_width - 2) + " ┃" for line in wrapped_lines) or \
             ("┃ " + "".ljust(box_width - 2) + " ┃")
  bottom = "┗" + "━" * box_width + "┛"

  return "\n".join([top, hline, sep, body, bottom])

def show_random_chunks(pages_and_texts: list, chunk_size: int = 500, k: int = 5, seed: int | None = 42):
    if seed is not None:
        random.seed(seed)

    all_chunks = chunk_pdf_pages(pages_and_texts, chunk_size=chunk_size)

    if not all_chunks:
        print("No chunks to display.")
        return

    idxs = _scattered_indices(len(all_chunks), k)

    print(f"showing{len(idxs)} scatttered random chunks out of {len(all_chunks)} total:\n")

    for i,idx in enumerate(idxs):
        print(f"#{i}")
        print(_draw_boxed_chunk(all_chunks[idx]))
        print()

#-----RUN---
assert 'pages_and_texts' in globals(), "pages_and_texts = open_and_read_pdf(pdf_path) first."
show_random_chunks(pages_and_texts, chunk_size=500, k=5, seed=42)

Recursive chunking

In [ ]:
from re import split
import nltk
from tqdm.auto import tqdm
nltk.download("punkt")

def recursive_chunk_text(text: str, max_chunk_size: int = 1000, min_chunk_size: int = 100) -> list:
    """
    Recursively splits a block of text into chunks that fit within size constraints.
    Tries splitting by sections, then newlines, then sentences.
    """

    def split_chunk(chunk: str) -> list:
        # Base case
        if len(chunk) <= max_chunk_size:
            return [chunk]

        # Try splitting by double newlines
        sections = chunk.split("\n\n")
        if len(sections) > 1:
            result = []
            for section in sections:
                if section.strip():
                    result.extend(split_chunk(section.strip()))
            return result

        # Try splitting by single newline
        sections = chunk.split("\n")
        if len(sections) > 1:
            result = []
            for section in sections:
                if section.strip():
                    result.extend(split_chunk(section.strip()))
            return result

         #Fallback: split by sentences

         sentences = nltk.sent_tokenize(chunk)
         chunks, current_chunk, current_size=[],[],0

         for sentence in sentences:
             if current_size=len(sentence)>max_chunk_size:
                 if current_chunk:
                     chunks.append(" ".join(current_chunk))
                  current_chunk=[sentence]
                  current_size=len(sentence)
             else:
                 current_chunk.append(sentence)
                 current_size+=len(sentence)
         if current_chunk:
             chunks.append(" ".join(current_chunk))
         return chunks
    return split_chunk(text)

def recursive_chunk_pdf_pages(pages_and_texts: list,
                              max_chunk_size: int = 1000,
                              min_chunk_size: int = 100) -> list[dict]:
    """
    Takes PDF pages with text and splits them into recursive chunks.

    Returns a list of dicts with page_number, chunk_index, and chunk_text.
    """
    all_chunks = []

    for page in tqdm(pages_and_texts, desc="Recursive chunking pages"):
        page_number = page["page_number"]
        page_text = page["text"]

        chunks = recursive_chunk_text(page_text,
                                      max_chunk_size=max_chunk_size,
                                      min_chunk_size=min_chunk_size)

        for i, chunk in enumerate(chunks):
            all_chunks.append({
                "page_number": page_number,
                "chunk_index": i,
                "chunk_char_count": len(chunk),
                "chunk_word_count": len(chunk.split()),
                "chunk_token_count": len(chunk) / 4,  # rough token estimate
                "chunk_text": chunk
            })

    return all_chunks




In [ ]:
from re import split
import nltk
from tqdm.auto import tqdm
nltk.download("punkt")

def recursive_chunk_text(text: str, max_chunk_size: int = 1000, min_chunk_size: int = 100) -> list:
    """
    Recursively splits a block of text into chunks that fit within size constraints.
    Tries splitting by sections, then newlines, then sentences.
    """

    def split_chunk(chunk: str) -> list:
        # Base case
        if len(chunk) <= max_chunk_size:
            return [chunk]

        # Try splitting by double newlines
        sections = chunk.split("\n\n")
        if len(sections) > 1:
            result = []
            for section in sections:
                if section.strip():
                    result.extend(split_chunk(section.strip()))
            return result

        # Try splitting by single newline
        sections = chunk.split("\n")
        if len(sections) > 1:
            result = []
            for section in sections:
                if section.strip():
                    result.extend(split_chunk(section.strip()))
            return result

        # Fallback: split by sentences
        sentences = nltk.sent_tokenize(chunk)
        chunks, current_chunk, current_size = [], [], 0

        for sentence in sentences:
            # Check if adding the next sentence exceeds the limit
            if current_size + len(sentence) > max_chunk_size:
                if current_chunk:
                    chunks.append(" ".join(current_chunk))
                current_chunk = [sentence]
                current_size = len(sentence)
            else:
                current_chunk.append(sentence)
                current_size += len(sentence)

        if current_chunk:
            chunks.append(" ".join(current_chunk))
        return chunks

    return split_chunk(text)

def recursive_chunk_pdf_pages(pages_and_texts: list,
                              max_chunk_size: int = 1000,
                              min_chunk_size: int = 100) -> list[dict]:
    """
    Takes PDF pages with text and splits them into recursive chunks.
    Returns a list of dicts with page_number, chunk_index, and chunk_text.
    """
    all_chunks = []

    for page in tqdm(pages_and_texts, desc="Recursive chunking pages"):
        page_number = page["page_number"]
        page_text = page["text"]

        chunks = recursive_chunk_text(page_text,
                                     max_chunk_size=max_chunk_size,
                                     min_chunk_size=min_chunk_size)

        for i, chunk in enumerate(chunks):
            all_chunks.append({
                "page_number": page_number,
                "chunk_index": i,
                "chunk_char_count": len(chunk),
                "chunk_word_count": len(chunk.split()),
                "chunk_token_count": len(chunk) / 4,  # rough token estimate
                "chunk_text": chunk
            })

    return all_chunks

In [ ]:
recursive_chunk_pages = recursive_chunk_pdf_pages(pages_and_texts,
                                                  max_chunk_size=800,
                                                  min_chunk_size=100)

print(f"Total recursive chunks: {len(recursive_chunk_pages)}")
print(f"First recursive chunk (page {recursive_chunk_pages[0]['page_number']}):")
print(recursive_chunk_pages[0]['chunk_text'][:200] + "...")

In [ ]:
import random,textwrap

def _scattered_indices(n:int,k:int,jitter_frac:float=0.08)->list[int]:

  if k<=0:
    return []

  if k==1:
    return [random.randrange(n)]

  anchors=[int(round(i*(n-1)/(k-1))) for i in range(k)]
  out,seen =[],set()
  radius=max(1,int(n*jitter_frac))
  for a in anchors:
    lo,hi= max(0, a-radius),min(n-1,a+radius)
    j = random.randint(lo,hi)
    if j not in seen:
      out.append(j); seen.add(j)
  while len(out)<k:
    r=random.randrange(n)
    if r not in seen:
      out.append(r);seen.add(r)
  return out

def _draw_boxed_chunk(c: dict, wrap_at:int=96)-> str:

  header=(
      f"chunk p{c['page_number']}.idx{c['chunk_index']}"
      f"chars{c['chunk_char_count']}.words {c['chunk_word_count']}. ~tokens {c['chunk_token_count']}"
  )

  wrapped_lines=textwrap.wrap(c['chunk_text'],width=wrap_at, break_long_words=False, replace_whitespace=False)

  content_width = max ([0, *map(len, wrapped_lines)])

  box_width = max(len(header), content_width+2)

  top    = "┏" + "━" * box_width + "┓"
  hline  = "┃" + header.ljust(box_width) + "┃"
  sep    = "┣" + "━" * box_width + "┫"
  body   = "\n".join("┃ " + line.ljust(box_width - 2) + " ┃" for line in wrapped_lines) or \
             ("┃ " + "".ljust(box_width - 2) + " ┃")
  bottom = "┗" + "━" * box_width + "┛"

  return "\n".join([top, hline, sep, body, bottom])

def show_random_chunks(pages_and_texts: list, chunk_size: int = 500, k: int = 5, seed: int | None = 42):
    if seed is not None:
        random.seed(seed)

    all_chunks = chunk_pdf_pages(pages_and_texts, chunk_size=chunk_size)

    if not all_chunks:
        print("No chunks to display.")
        return

    idxs = _scattered_indices(len(all_chunks), k)

    print(f"showing{len(idxs)} scatttered random chunks out of {len(all_chunks)} total:\n")

    for i,idx in enumerate(idxs):
        print(f"#{i}")
        print(_draw_boxed_chunk(all_chunks[idx]))
        print()

#-----RUN---
assert 'pages_and_texts' in globals(), "pages_and_texts = open_and_read_pdf(pdf_path) first."
show_random_chunks(pages_and_texts, chunk_size=500, k=5, seed=42)

Structural chunking

In [ ]:
import re
import random
import textwrap

# 1) Helper to detect "chapter start" pages
def _is_chapter_header_page(text: str) -> bool:
    # Robust to punctuation/diacritics differences; matches the recurring header
    # e.g., "UNIVERSITY OF HAWAI'I AT MĀNOA FOOD SCIENCE AND HUMAN NUTRITION PROGRAM"
    return re.search(r"university\s+of\s+hawai", text, flags=re.IGNORECASE) is not None

def _guess_title_from_page(text: str) -> str:
    """
    Best-effort chapter title guess = the text before the 'University of Hawai' header line.
    Falls back to the first ~120 characters.
    """
    m = re.search(r"university\s+of\s+hawai", text, flags=re.IGNORECASE)
    if m:
        title = text[:m.start()].strip()
        # keep it readable
        title = re.sub(r"\s+", " ", title).strip()
        if 10 <= len(title) <= 180:
            return title

    # fallback
    t = re.sub(r"\s+", " ", text).strip()
    return t[:120] if t else "Untitled Chapter"

# 2) Build chapter chunks
def chapter_chunk_pdf_pages(pages_and_texts: list[dict]) -> list[dict]:
    """
    Returns a list of chapter chunks:
    [
        {
            'chapter_index': int,
            'title': str,
            'page_start': int,  # adjusted page number (your -41 offset)
            'page_end': int,
            'chunk_char_count': int,
            'chunk_word_count': int,
            'chunk_token_count': float,  # ~chars/4
            'chunk_text': str
        }, ...
    ]
    """
    if not pages_and_texts:
        return []

    # Find all page indices that look like the start of a chapter
    chapter_starts = []
    for i, p in enumerate(pages_and_texts):
        txt = p["text"]
        if _is_chapter_header_page(txt):
            chapter_starts.append(i)

    # If nothing detected, return empty (or treat entire doc as one chunk)
    if not chapter_starts:
        # Treat entire doc as one "chapter"
        all_text = " ".join(p["text"] for p in pages_and_texts).strip()
        return [{
            "chapter_index": 0,
            "title": _guess_title_from_page(pages_and_texts[0]["text"]),
            "page_start": pages_and_texts[0]["page_number"],
            "page_end": pages_and_texts[-1]["page_number"],
            "chunk_char_count": len(all_text),
            "chunk_word_count": len(all_text.split()),
            "chunk_token_count": round(len(all_text) / 4, 2),
            "chunk_text": all_text
        }]

    # Build chapter ranges (start -> next_start-1)
    chapter_chunks = []
    for ci, s in enumerate(chapter_starts):
        e = (chapter_starts[ci + 1] - 1) if (ci + 1 < len(chapter_starts)) else (len(pages_and_texts) - 1)
        if e < s:
            continue  # guard (shouldn't happen)

        pages = pages_and_texts[s:e + 1]
        text_concat = " ".join(p["text"] for p in pages).strip()
        title = _guess_title_from_page(pages[0]["text"])

        chapter_chunks.append({
            "chapter_index": ci,
            "title": title,
            "page_start": pages[0]["page_number"],
            "page_end": pages[-1]["page_number"],
            "chunk_char_count": len(text_concat),
            "chunk_word_count": len(text_concat.split()),
            "chunk_token_count": round(len(text_concat) / 4, 2),
            "chunk_text": text_concat
        })

    return chapter_chunks


In [ ]:
structure_chunked_pages = chapter_chunk_pdf_pages(pages_and_texts)

print(f"Total chapter-based chunks: {len(structure_chunked_pages)}")
if structure_chunked_pages:
    first = structure_chunked_pages[0]
    print(f"First chapter (pages {first['page_start']}-{first['page_end']}): {first['title']}")
    print(print(first['chunk_text'][:200] + "..."))
else:
    print("No chapters detected.")

In [ ]:
def _draw_boxed_chunk(c: dict, wrap_at: int = 96) -> str:
    header = (
        f" Chapter {c.get('chapter_index', '?')} | {c.get('title', 'Untitled')[:120]} "
        f"| p{c.get('page_start', '?')}-{c.get('page_end', '?')} | ~tokens {c.get('chunk_token_count', 0)}"
    )
    wrapped = textwrap.wrap(c["chunk_text"], width=wrap_at, break_long_words=False, replace_whitespace=False)
    content_width = max(len(header), *(len(x) for x in wrapped)) if wrapped else len(header)

    top    = "┏" + "━" * content_width + "┓"
    hline  = "┃" + header.ljust(content_width) + "┃"
    sep    = "┣" + "━" * content_width + "┫"
    body   = "\n".join("┃ " + line.ljust(content_width-2) + " ┃" for line in wrapped[:12])
    bottom = "┗" + "━" * content_width + "┛"

    return "\n".join([top, hline, sep, body, bottom])

def show_random_chapter_chunks(chapter_chunks, k=5, seed=42):
    if not chapter_chunks: return print("No chunks to display.")
    if seed is not None: random.seed(seed)

    idxs = random.sample(range(len(chapter_chunks)), min(k, len(chapter_chunks)))
    print(f"Showing {len(idxs)} random chapters out of {len(chapter_chunks)} total:\n")
    for i, idx in enumerate(idxs, 1):
        print(f"#{i}")
        print(_draw_boxed_chunk(chapter_chunks[idx]))
        print()

# 4) Run
assert 'pages_and_texts' in globals(), "Run your base PDF loader first to define `pages_and_texts`."

chapter_chunks = chapter_chunk_pdf_pages(pages_and_texts)
print(f"Total chapters detected: {len(chapter_chunks)}")

if chapter_chunks:
    first = chapter_chunks[0]
    print(f"First chapter: {first['title']} (p{first['page_start']}-{first['page_end']})")

# Inspect a few
show_random_chapter_chunks(chapter_chunks, k=5, seed=21)

LLM based Chunking

In [ ]:
import os
os.environ['OPENAI_API_KEY']=

In [ ]:
from openai import OpenAI
from typing import List, Dict
from tqdm.auto import tqdm

# Initialize OpenAI client
client = OpenAI()

def llm_based_chunk(text: str, chunk_size: int = 1000, model: str = "gpt-4o-mini") -> List[str]:
    """
    Uses an LLM to find semantically coherent chunk boundaries around a target size.
    """
    def get_chunk_boundary(text_segment: str) -> int:
        prompt = f"""
Analyze the following text and identify the best point to split it into two semantically coherent parts.
The split should occur near {chunk_size} characters.

Text:
\"\"\"{text_segment}\"\"\"

Return only the integer index (character position) within this text where the split should occur.
Do not return any explanation.
"""
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are a text analysis expert."},
                {"role": "user", "content": prompt}
            ],
            temperature=0
        )

        # Extract and sanitize
        split_str = response.choices[0].message.content.strip()
        try:
            split_point = int(split_str)
        except ValueError:
            split_point = chunk_size
        return split_point

    chunks = []
    remaining_text = text

    while len(remaining_text) > chunk_size:
        # Look ahead twice the chunk size to give the LLM context for the split
        text_window = remaining_text[:chunk_size * 2]
        split_point = get_chunk_boundary(text_window)

        # Safety check: ensure split isn't too small or larger than window
        if split_point < 100 or split_point > len(text_window) - 100:
            split_point = chunk_size

        chunks.append(remaining_text[:split_point].strip())
        remaining_text = remaining_text[split_point:].strip()

    if remaining_text:
        chunks.append(remaining_text)

    return chunks

def llm_based_chunk_pdf_pages(pages_and_texts: List[Dict], chunk_size: int = 1000, model: str = "gpt-4o-mini") -> List[Dict]:
    """Applies LLM-based chunking to each PDF page."""
    all_chunks = []
    for page in tqdm(pages_and_texts, desc="LLM-based chunking pages"):
        page_number = page["page_number"]
        page_text = page["text"]

        chunks = llm_based_chunk(page_text, chunk_size=chunk_size, model=model)
        for i, chunk in enumerate(chunks):
            all_chunks.append({
                "page_number": page_number,
                "chunk_index": i,
                "chunk_char_count": len(chunk),
                "chunk_word_count": len(chunk.split()),
                "chunk_token_count": len(chunk) / 4, # rough estimate
                "chunk_text": chunk
            })
    return all_chunks

In [ ]:
llm_chunked_pages=llm_based_chunk_pdf_pages(pages_and_texts,
                                            chunk_size=800,
                                            model='gpt-4o-mini'
                                      )
print(f"Total LLM-based chunks:{len(llm_chunked_pages)}")
print(f"First LLM-based chunk (page {llm_chunked_pages[0]['page_number']}:")
print(llm_chunked_pages[0]['chunk_text'][:200] + "...")


In [ ]:
import textwrap
import random

def _draw_boxed_chunk(c: dict, wrap_at: int = 96) -> str:
    """Creates a Unicode box around chunk text for terminal viewing."""
    header = (
        f" Chapter {c['chapter_index']} | {c['title'][:120]} "
        f"| p{c['page_start']}-{c['page_end']} | ~tokens {c.get('chunk_token_count', 'N/A')}"
    )
    wrapped = textwrap.wrap(c["chunk_text"], width=wrap_at, break_long_words=False, replace_whitespace=False)
    content_width = max(len(header), *(len(x) for x in wrapped)) if wrapped else len(header)

    top    = "┏" + "━" * content_width + "┓"
    hline  = "┃" + header.ljust(content_width) + "┃"
    sep    = "┣" + "━" * content_width + "┫"
    body   = "\n".join("┃ " + line.ljust(content_width-2) + " ┃" for line in wrapped[:12]) # Show first 12 lines
    bottom = "┗" + "━" * content_width + "┛"

    return "\n".join([top, hline, sep, body, bottom])

def show_random_chapter_chunks(chapter_chunks: List[Dict], k: int = 5, seed: int | None = 42):
    """Prints a random selection of chunks for quality inspection."""
    if not chapter_chunks:
        print("No chapter chunks to display."); return
    if seed is not None:
        random.seed(seed)

    idxs = random.sample(range(len(chapter_chunks)), min(k, len(chapter_chunks)))
    print(f"Showing {len(idxs)} random chapters out of {len(chapter_chunks)} total:\n")
    for i, idx in enumerate(idxs, 1):
        print(f"#{i}")
        print(_draw_boxed_chunk(chapter_chunks[idx]))
        print()


# Assuming pages_and_texts is already defined from your PDF loader
chapter_chunks = chapter_chunk_pdf_pages(pages_and_texts)
print(f"Total chapters detected: {len(chapter_chunks)}")

# Inspect a few random results
show_random_chapter_chunks(chapter_chunks, k=3, seed=21)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- Config: choose which size metric to analyze ----
# Options: "chars" (chunk_char_count), "words" (chunk_word_count), "tokens" (chunk_token_count)
METRIC = "words"

def _size_val(c, metric: str):
    if metric == "chars":
        return c.get("chunk_char_count", len(c.get("chunk_text", "")))
    if metric == "words":
        # Returns existing word count or calculates it on the fly
        return c.get("chunk_word_count", len(c.get("chunk_text", "").split()))
    if metric == "tokens":
        # fall back to chars/4 if not present
        return c.get("chunk_token_count", len(c.get("chunk_text", ""))/4)
    raise ValueError(f"METRIC must be one of {{'chars', 'words', 'tokens'}}")

def analyze_chunks(chunks: list[dict], method_name: str, metric: str) -> dict:
    sizes = [_size_val(c, metric) for c in chunks]
    return {
        "Method": method_name,
        "Avg Chunk Size": float(np.mean(sizes)) if sizes else 0.0,
        "Num Chunks": len(sizes),
        "Size Variance": float(np.var(sizes)) if sizes else 0.0,
    }

# ---- Gather results from the previously computed lists ----
# Ensure these lists were defined in earlier steps of your notebook
datasets = {
    "fixed": chunked_pages,
    "semantic": semantic_chunked_pages,
    "recursive": recursive_chunk_pages,
    "structure": structure_chunked_pages,
    # "llm": llm_chunked_pages,
}

# Run the analysis
results = [analyze_chunks(data, name, METRIC) for name, data in datasets.items() if data]
df_results = pd.DataFrame(results)
print(df_results)

print("\n# Performance analysis")
print("1. Structure-based chunking produced the most coherent sections.")
print("2. Semantic chunking maintained best context preservation.")
print("3. Fixed-size chunking showed lowest variance but poor semantic coherence.")
print("4. Recursive chunking provided balanced results.")
print("5. LLM chunking provided balanced results.")

In [ ]:
flg,axes =plt.subplots(1,3,figsize=(18,5))

axes[0].bar(df_results['Method'],df_results['Avg Chunk Size'])
axes[0].set_title('Average Chunk Size')
axes[0].set_ylabel({'chars':"Characters","words":"Words","tokens":"Tokens"}[METRIC])

axes[1].bar(['Method'],df_results['Num Chunks'])
axes[1].set_title('Number of Chunks')
axes[1].set_ylabel('count')


axes[2].bar(df_results['Method'],df_results['Size Variance'])
axes[2].set_title('Chunk Size Variance')
axes[2].set_ylabel('variance')

plt.suptitle(f"Chunking Method Comparision (metric:{METRIC})")
plt.tight_layout()
plt.show()

def extract_sizes(chunks,metric:str):
  return [_size_val(c,metric) for c in chunks]

sizes_fixed=extract_sizes(chunked_pages,METRIC)
sizes_semantic=extract_sizes(semantic_chunked_pages,METRIC)
sizes_recursive=extract_sizes(recursive_chunk_pages,METRIC)
sizes_structure=extract_sizes(structure_chunked_pages,METRIC)
# sizes_llm=extract_sizes(llm_chunked_pages,METRIC)


plt.figure(figsize=(10,6))
plt.boxplot(
    [sizes_fixed, sizes_semantic, sizes_recursive, sizes_structure],
    label=['Fixed', 'Semantic', 'Recursive', 'Structure'],
    patch_artist=True
)

plt.title(f"Chunk Size Distribution (boxplot)(metric:{METRIC})")
plt.ylabel({'chars':"Characters","words":"Words","tokens":"Tokens"}[METRIC])
plt.tight_layout()
plt.show()
